# Example ETL pipeline for the MDC raw data

ngl don't even read this bull

In [25]:
import duckdb 
import pandas as pd

In [26]:
# Convert to more consistent object
df = pd.read_csv("data/Mtl_Tor_Edm_Van_CSDs_Natasha.csv", encoding='latin-1', skiprows=4, header=0) # Discard metadata rows
df.to_parquet("data/Mtl_Tor_Edm_Van_CSDs_Natasha.parquet") # Force persistent UTF-8 encoding

## Extracting the raw data 

In [27]:
df = pd.read_parquet("data/Mtl_Tor_Edm_Van_CSDs_Natasha.parquet") # Overwrite old .csv read
df.head(10)

,Unnamed: 0,Unnamed: 1,Total - Tenure,Total - Tenure.1,Total - Tenure.2,Owner,Owner.1,Owner.2,Renter,Renter.1,Renter.2
0,Geography,Immigrant Status,Total population,Average total income of household ($),Average STIR before taxes (%),Total population,Average total income of household ($),Average STIR before taxes (%),Total population,Average total income of household ($),Average STIR before taxes (%)
1,Canada 20000 ( 4.3%),Total Immigrant Status,34973090,131000,19.2,25209770,148800,17.3,9763320,84900,24
2,Canada 20000 ( 4.3%),Non-immigrants,23648635,131400,17.7,17535505,149200,15.6,6113130,80800,23.8
3,Canada 20000 ( 4.3%),Immigrant,11324455,129800,22.1,7674270,148000,21.1,3650190,91700,24.4
4,"Montréal (CMA), Que. 20000 ( 3.4%)",Total Immigrant Status,4133840,122900,18.3,2566025,150800,16.3,1567815,77300,21.5
5,"Montréal (CMA), Que. 20000 ( 3.4%)",Non-immigrants,2644045,129200,17.2,1769440,155800,14.9,874610,75300,21.9
6,"Montréal (CMA), Que. 20000 ( 3.4%)",Immigrant,1489795,111700,20.1,796590,139400,19.3,693205,79900,21.1
7,Lavaltrie (2452007) V 00000 ( 3.0%),Total Immigrant Status,14225,107900,16.5,11475,116700,15.2,2745,71100,22
8,Lavaltrie (2452007) V 00000 ( 3.0%),Non-immigrants,13555,108200,16.4,10870,117300,15.1,2680,71300,22
9,Lavaltrie (2452007) V 00000 ( 3.0%),Immigrant,665,103200,18.1,610,107200,17.6,60,62000,23.6


This is a pivot table - we essentially need to unpivot.

## Transforming the data

In [28]:
df.iloc[0].values # These need to become the new headers

<ArrowStringArray>
[                            'Geography',
                      'Immigrant Status',
                      'Total population',
 'Average total income of household ($)',
         'Average STIR before taxes (%)',
                      'Total population',
 'Average total income of household ($)',
         'Average STIR before taxes (%)',
                      'Total population',
 'Average total income of household ($)',
         'Average STIR before taxes (%)']
Length: 11, dtype: str

In [29]:
cols = df.iloc[0]
df.columns = cols
df.drop(0, inplace=True)
df.head()

,Geography,Immigrant Status,Total population,Average total income of household ($),Average STIR before taxes (%),Total population,Average total income of household ($),Average STIR before taxes (%),Total population,Average total income of household ($),Average STIR before taxes (%)
1,Canada 20000 ( 4.3%),Total Immigrant Status,34973090,131000,19.2,25209770,148800,17.3,9763320,84900,24
2,Canada 20000 ( 4.3%),Non-immigrants,23648635,131400,17.7,17535505,149200,15.6,6113130,80800,23.8
3,Canada 20000 ( 4.3%),Immigrant,11324455,129800,22.1,7674270,148000,21.1,3650190,91700,24.4
4,"Montréal (CMA), Que. 20000 ( 3.4%)",Total Immigrant Status,4133840,122900,18.3,2566025,150800,16.3,1567815,77300,21.5
5,"Montréal (CMA), Que. 20000 ( 3.4%)",Non-immigrants,2644045,129200,17.2,1769440,155800,14.9,874610,75300,21.9


We've discarded the `Total - Tenure` / `Owner` / `Renter` labels. We need to add them back.

In [30]:
# Reset index to make manipulation easier
df = df.reset_index(drop=True)
df.head()

,Geography,Immigrant Status,Total population,Average total income of household ($),Average STIR before taxes (%),Total population,Average total income of household ($),Average STIR before taxes (%),Total population,Average total income of household ($),Average STIR before taxes (%)
0,Canada 20000 ( 4.3%),Total Immigrant Status,34973090,131000,19.2,25209770,148800,17.3,9763320,84900,24
1,Canada 20000 ( 4.3%),Non-immigrants,23648635,131400,17.7,17535505,149200,15.6,6113130,80800,23.8
2,Canada 20000 ( 4.3%),Immigrant,11324455,129800,22.1,7674270,148000,21.1,3650190,91700,24.4
3,"Montréal (CMA), Que. 20000 ( 3.4%)",Total Immigrant Status,4133840,122900,18.3,2566025,150800,16.3,1567815,77300,21.5
4,"Montréal (CMA), Que. 20000 ( 3.4%)",Non-immigrants,2644045,129200,17.2,1769440,155800,14.9,874610,75300,21.9


In [31]:
# Unpivot tenure types and measurements into long format
tenure_types = ['Total - Tenure', 'Owner', 'Renter']
metrics = ['Total population', 'Average total income of household ($)', 'Average STIR before taxes (%)']

# Step 1: Split by tenure type using iloc for positional selection
dfs_list = []
for i, tenure in enumerate(tenure_types):
    col_start = 2 + i * 3
    # Use iloc to select columns by position: Geography(0), Immigrant Status(1), then 3 metrics
    df_temp = df.iloc[:, [0, 1, col_start, col_start+1, col_start+2]].copy()
    df_temp.columns = ['Geography', 'Immigrant Status'] + metrics
    df_temp['Tenure Type'] = tenure
    
    dfs_list.append(df_temp)

df_unpivoted = pd.concat(dfs_list, ignore_index=True)

# Step 2: Unpivot measurements into single column
df_long = df_unpivoted.melt(
    id_vars=['Geography', 'Immigrant Status', 'Tenure Type'],
    value_vars=metrics,
    var_name='Measurement',
    value_name='Value'
)

# Clean up
df_long['Value'] = pd.to_numeric(df_long['Value'], errors='coerce')
df_long['Immigrant Status'] = df_long['Immigrant Status'].replace('Total Immigrant Status', 'Total')

df_long.head(15)

,Geography,Immigrant Status,Tenure Type,Measurement,Value
0,Canada 20000 ( 4.3%),Total,Total - Tenure,Total population,34973090.0
1,Canada 20000 ( 4.3%),Non-immigrants,Total - Tenure,Total population,23648635.0
2,Canada 20000 ( 4.3%),Immigrant,Total - Tenure,Total population,11324455.0
3,"Montréal (CMA), Que. 20000 ( 3.4%)",Total,Total - Tenure,Total population,4133840.0
4,"Montréal (CMA), Que. 20000 ( 3.4%)",Non-immigrants,Total - Tenure,Total population,2644045.0
5,"Montréal (CMA), Que. 20000 ( 3.4%)",Immigrant,Total - Tenure,Total population,1489795.0
6,Lavaltrie (2452007) V 00000 ( 3.0%),Total,Total - Tenure,Total population,14225.0
7,Lavaltrie (2452007) V 00000 ( 3.0%),Non-immigrants,Total - Tenure,Total population,13555.0
8,Lavaltrie (2452007) V 00000 ( 3.0%),Immigrant,Total - Tenure,Total population,665.0
9,Richelieu (2455057) V 00000 ( 2.1%),Total,Total - Tenure,Total population,5335.0


In [32]:
# Save to DuckDB database
con = duckdb.connect("data/mdc_raw_data.duckdb")

# Create table from dataframe (overwrites if exists)
con.execute("DROP TABLE IF EXISTS census_data")
con.execute("""
    CREATE TABLE census_data AS 
    SELECT * FROM df_long
""")

# Verify
print("Tables:", con.execute("SHOW TABLES").fetchall())
print("Row count:", con.execute("SELECT COUNT(*) FROM census_data").fetchone()[0])

# Sample queries
print("\n--- Sample Query: Montréal CMA population by immigrant status ---")
result = con.execute("""
    SELECT Geography, "Immigrant Status", "Tenure Type", Measurement, Value
    FROM census_data
    WHERE Geography LIKE 'Montréal%' 
      AND Measurement = 'Total population'
      AND "Tenure Type" = 'Total - Tenure'
""").fetchdf()
print(result)

print("\n--- Sample Query: All subdivisions of Montréal CMA ---")
result = con.execute("""
    SELECT DISTINCT Geography
    FROM census_data
    WHERE Geography LIKE 'Montréal%'
    ORDER BY Geography
""").fetchdf()
print(result)

con.close()

Tables: [('census_data',)]
Row count: 5238

--- Sample Query: Montréal CMA population by immigrant status ---
Empty DataFrame
Columns: [Geography, Immigrant Status, Tenure Type, Measurement, Value]
Index: []

--- Sample Query: All subdivisions of Montréal CMA ---
Empty DataFrame
Columns: [Geography]
Index: []
